In [ ]:
import csky_king_experimental as cy #load the NTv5p4 branch of csky
import numpy as np
from kingmaker.wrapper import KingSpatialLikelihood

In [ ]:
timer = cy.timing.Timer()
time = timer.time

In [ ]:
mp_cpus= 8

gamma = 2.5

## Load analysis

In [5]:
with time('load analysis...'):
    ana_dir = cy.utils.ensure_dir('/data/user/fkrafft/csky_king_tests')
    ana = cy.get_analysis(cy.selections.repo, 'version-005-p04', cy.selections.NTDataSpecs.NTv5p4, dir = ana_dir, min_sigma = np.radians(0.2))

Setting up Analysis for:
NTv5p4_IC79, NTv5p4_IC86
Setting up NTv5p4_IC79...


Energy PDF Ratio Model...
  * gamma = 4.0000 ...
Signal Acceptance Model...
  * gamma = 4.0000 ...
not applying any masking
Setting up NTv5p4_IC86...
Reading /data/ana/analyses/northern_tracks/version-005-p04/IC86_pass2_MC.npy ...
Reading /data/ana/analyses/northern_tracks/version-005-p04/IC86_2011_exp.npy ...
Reading /data/ana/analyses/northern_tracks/version-005-p04/IC86_2012_exp.npy ...
Reading /data/ana/analyses/northern_tracks/version-005-p04/IC86_2013_exp.npy ...
Reading /data/ana/analyses/northern_tracks/version-005-p04/IC86_2014_exp.npy ...
Reading /data/ana/analyses/northern_tracks/version-005-p04/IC86_2015_exp.npy ...
Reading /data/ana/analyses/northern_tracks/version-005-p04/IC86_2016_exp.npy ...
Reading /data/ana/analyses/northern_tracks/version-005-p04/IC86_2017_exp.npy ...
Reading /data/ana/analyses/northern_tracks/version-005-p04/IC86_2018_exp.npy ...
Reading /data/ana/analyses/northern_tracks/version-005-p04/IC86_2019_exp.npy ...
Reading /data/ana/analyses/northern_trac

## Initialize Bins

In [8]:
parametrization_bins = {
        'sindec': 10,
        'log10energy': 10,
        'sigma': 10,
        }

## Initialize Likelihood Wrappers

- The likelihood wrapppers fit the provided signal events with King function based on provided parametrization bins on initialization.
- 

In [9]:
king_space_eval_79 = KingSpatialLikelihood(signal_events = ana[0].sig.as_array,
                                           parametrization_bins = parametrization_bins,
                                           cache_name = "/data/user/bbrinson/lrd/king_caches/NTv5p4_IC79_cache.npz",
                                           weight_field = "oneweight",
                                           true_ra_name = "true_ra",
                                           true_dec_name = "true_dec",
                                           true_energy_name = "true_energy",
                                           angular_cutoff = np.radians(15),
                                           )

king_space_eval_86 = KingSpatialLikelihood(signal_events = ana[1].sig.as_array,
                                           parametrization_bins = parametrization_bins,
                                           cache_name = "/data/user/bbrinson/lrd/king_caches/NTv5p4_IC86_cache.npz",
                                           weight_field = "oneweight",
                                           true_ra_name = "true_ra",
                                           true_dec_name = "true_dec",
                                           true_energy_name = "true_energy",
                                           angular_cutoff = np.radians(15),
                                           )

In [12]:
def king_func_79(ev, pdf_bg, src, gamma, **kwargs):
    king_space_eval_79.set_events(events = ev, source_ras = src.ra, source_decs = src.dec)
    pdf_ratio = king_space_eval_79.evaluate_pdf(events = ev, gamma = gamma)
    pdf_ratio /= pdf_bg
    return pdf_ratio

def king_func_86(ev, pdf_bg, src, gamma, **kwargs):
    king_space_eval_86.set_events(events = ev, source_ras = src.ra, source_decs = src.dec)
    pdf_ratio = king_space_eval_86.evaluate_pdf(events = ev, gamma = gamma)
    pdf_ratio /= pdf_bg
    return pdf_ratio


## Prepare trial runner

In [13]:
dtype = [('gamma', '<f8'), ('ns', '<f8'), ('ts', '<f8')]

ras_deg = [40.67]
decs_deg = [-0.01]

ras = np.radians(ras_deg)
decs = np.radians(decs_deg)

# Configure trial runner
cy.CONF['mp_cpus'] = mp_cpus
mask_deg = 15.

for i, dec in enumerate(decs):
    ra = ras[i]
    srcs = cy.sources(ra, dec)

    tr = cy.get_trial_runner(ana = ana,
                             src = srcs,
                             use_bdt = True,
                             flux = cy.hyp.PowerLawFlux(gamma),
                             use_all_ev = True,
                             use_pdf_bg = True,
                             space = 'generic',
                             func_array = [king_func_79, king_func_86],
                             extra_keep = ["dec", "ra", "sigma", "sigma_bdt", "sindec", "event", "energy"],
                             cut_n_sigma = np.inf,
                             )

    # Run and save trials per declination
    bg_trials = []


## Run background trials (don't do that to cobalt :D)

In [ ]:
""" with time('run trials'):
    for j in range(N_trials):
        temp_seed = seed * N_trials + j
        trial = tr.get_one_trial(n_sig = 0, seed = temp_seed)
        #masks = [dist_mask(evs[0], srcs, mask_deg) for evs in trial.evss]
        #fit = tr.get_one_fit_from_trial(trial, _cut_deg = mask_deg)
        fit = tr.get_one_fit_from_trial(trial)
        bg_trials.append(fit)

bg_trials = np.array(bg_trials)

new_bg_array = np.zeros(len(bg_trials), dtype = dtype)
new_bg_array['gamma'] = bg_trials[:, 2]
new_bg_array['ns'] = bg_trials[:, 1]
new_bg_array['ts'] = bg_trials[:, 0] """

In [ ]:
""" bg_dir = cy.utils.ensure_dir('/data/user/bbrinson/lrd/king_tests/bg')
    np.save('{}/N_{}_seed_{}.npy'.format(bg_dir, N_trials, seed), new_bg_array) """